#Sistema concurrente de predicción de depresión posparto con K-means en Go

# **Objetivo:** Procesar PPD dataset con máxima eficiencia


##Dependencias, utilizamos la API de Kaggle

In [1]:
# Instalar librerías necesarias
!pip install -q polars pyarrow kaggle dask requests psutil

print("Correcto")

Correcto


In [2]:
from google.colab import files
import os
from pathlib import Path

print("CONFIGURAR KAGGLE API")

#directorio .kaggle
kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)

print("\n1️archivo kaggle.json")
print("Descárgalo de: https://www.kaggle.com/settings/account)")
print("\n📁Selecciona kaggle.json...")

uploaded_files = files.upload()

#ubicación correcta
for filename in uploaded_files.keys():
    if filename == 'kaggle.json':
        os.rename(filename, str(kaggle_dir / 'kaggle.json'))
        os.chmod(str(kaggle_dir / 'kaggle.json'), 0o600)
        print(f"kaggle.json configurado correctamente")
    else:
        os.remove(filename)

print(f"\n📍 Ruta: {kaggle_dir / 'kaggle.json'}")

CONFIGURAR KAGGLE API

1️archivo kaggle.json
Descárgalo de: https://www.kaggle.com/settings/account)

📁Selecciona kaggle.json...


Saving kaggle.json to kaggle.json
kaggle.json configurado correctamente

📍 Ruta: /root/.kaggle/kaggle.json


In [3]:
import subprocess
import os

print("DESCARGAR DATASET DESDE KAGGLE")

#directorio para dataset
os.makedirs('data', exist_ok=True)

print("\Descargando dataset...")
print("(Sin abrir navegador, sin Google Drive)\n")

# Descargar usando Kaggle CLI
result = subprocess.run(
    ['kaggle', 'datasets', 'download', '-d',
     'parvezalmuqtadir2348/postpartum-depression',
     '-p', 'data', '--unzip'],
    capture_output=True,
    text=True
)
if result.returncode == 0:
    print("Descargado exitosamente")
    # Listar archivos descargados
    files_list = os.listdir('data')
    print(f"\nArchivos descargados: {len(files_list)}")
    for f in files_list[:5]:
        size = os.path.getsize(f'data/{f}') / (1024**2)
        print(f"   • {f} ({size:.1f} MB)")
else:
    print(f"Error: {result.stderr}")

<>:9: SyntaxWarning: invalid escape sequence '\D'
<>:9: SyntaxWarning: invalid escape sequence '\D'
/tmp/ipykernel_2791/2182593382.py:9: SyntaxWarning: invalid escape sequence '\D'
  print("\Descargando dataset...")


DESCARGAR DATASET DESDE KAGGLE
\Descargando dataset...
(Sin abrir navegador, sin Google Drive)

Descargado exitosamente

Archivos descargados: 1
   • post natal data.csv (0.1 MB)


##Cargar con Polars (Lazy Evaluation)

In [4]:
import polars as pl
import psutil

print("CARGAR DATASET CON POLARS")

# Encontrar el CSV
csv_files = [f for f in os.listdir('data') if f.endswith('.csv')]
csv_path = f'data/{csv_files[0]}'

print(f"\nArchivo: {csv_files[0]}")
print(f"Tamaño: {os.path.getsize(csv_path) / (1024**2):.1f} MB")

print("\nOPCIÓN A: scan_csv() - LAZY (SIN cargar en memoria)")
start_memory = psutil.Process().memory_info().rss / (1024**2)
print(f"RAM antes: {start_memory:.1f} MB")

# Lazy: solo lee headers, crea plan de ejecución
df_lazy = pl.scan_csv(csv_path)

current_memory = psutil.Process().memory_info().rss / (1024**2)
print(f"RAM después: {current_memory:.1f} MB")
print(f"✓ Datos NO cargados en memoria")
print(f"✓ Solo plan de ejecución ({(current_memory - start_memory):.1f} MB)")

print("\nINFORMACIÓN DEL SCHEMA:")
print(df_lazy.collect_schema())

CARGAR DATASET CON POLARS

Archivo: post natal data.csv
Tamaño: 0.1 MB

OPCIÓN A: scan_csv() - LAZY (SIN cargar en memoria)
RAM antes: 128.2 MB
RAM después: 129.1 MB
✓ Datos NO cargados en memoria
✓ Solo plan de ejecución (0.9 MB)

INFORMACIÓN DEL SCHEMA:
Schema({'Timestamp': String, 'Age': String, 'Feeling sad or Tearful': String, 'Irritable towards baby & partner': String, 'Trouble sleeping at night': String, 'Problems concentrating or making decision': String, 'Overeating or loss of appetite': String, 'Feeling anxious': String, 'Feeling of guilt': String, 'Problems of bonding with baby': String, 'Suicide attempt': String})


##Autodetectar estructura

In [5]:
print("DETECTANDO ESTRUCTURA DEL DATASET")

# Clasificar columnas por tipo
schema = df_lazy.collect_schema()
numeric_cols = [name for name, dtype in schema.items()
                if str(dtype) in ['Int64', 'Int32', 'Float64', 'Float32']]
string_cols = [name for name, dtype in schema.items()
              if str(dtype) == 'String']

print(f"\n✓ Columnas numéricas: {len(numeric_cols)}")
for nombre_col in numeric_cols:
    print(f"   • {nombre_col}")

print(f"\n✓ Columnas de texto (categóricas): {len(string_cols)}")
for nombre_col in string_cols:
    print(f"   • {nombre_col}")

# Variable target: se define explícitamente. Antes se tomaba la primera columna con
# 3 valores o menos, que coincidía con esta solo por el orden de las columnas.
target_col = 'Feeling sad or Tearful'
assert target_col in string_cols, f"No se encontró la columna target '{target_col}'"
print(f"\n✅ TARGET: '{target_col}'")

DETECTANDO ESTRUCTURA DEL DATASET

✓ Columnas numéricas: 0

✓ Columnas de texto (categóricas): 11
   • Timestamp
   • Age
   • Feeling sad or Tearful
   • Irritable towards baby & partner
   • Trouble sleeping at night
   • Problems concentrating or making decision
   • Overeating or loss of appetite
   • Feeling anxious
   • Feeling of guilt
   • Problems of bonding with baby
   • Suicide attempt

✅ TARGET: 'Feeling sad or Tearful'


##Transformaciones Lazy (SIN ejecutar aún)

In [6]:
import polars as pl

print("DEFINIR TRANSFORMACIONES LAZY (aún NO se ejecutan)")

# Codificación explícita de cada respuesta. replace_strict lanza un error si aparece una
# respuesta no prevista, en vez de convertirla en 0 ("No") sin avisar.
EDAD_A_NUMERO = {'18-20': 19.0, '20-25': 22.5, '25-30': 27.5, '30-35': 32.5,
                 '35-40': 37.5, '40-45': 42.5, '45-50': 47.5, '50-55': 52.5}
RESPUESTA_A_NUMERO = {
    'No': 0.0, 'Not at all': 0.0,
    # Respuestas intermedias de frecuencia o duda
    'Sometimes': 0.5, 'Maybe': 0.5, 'Often': 0.5, 'Two or more days a week': 0.5,
    'Yes': 1.0,
}
# 'Not interested to say' (Suicide attempt) no equivale a "No": se trata como dato faltante
NO_RESPONDE = 'Not interested to say'

sintomas = [c for c in string_cols if c not in ['Timestamp', 'Age']]
columnas_numericas = ['Age_numeric'] + [f"{c}_encoded" for c in sintomas]

print("\n1️CONVERSIÓN DE TIPOS: STRING → NÚMERO")
print("   (Age: '35-40' → 37.5 | No/Not at all→0, Sometimes/Maybe/Often/Two or more days→0.5, Yes→1)")
df_transformed = df_lazy.with_columns(
    pl.col('Age').replace_strict(EDAD_A_NUMERO, return_dtype=pl.Float64).alias('Age_numeric'),
    *[
        pl.when(pl.col(c) == NO_RESPONDE).then(pl.lit(None, dtype=pl.String)).otherwise(pl.col(c))
        .replace_strict(RESPUESTA_A_NUMERO, return_dtype=pl.Float64)
        .alias(f"{c}_encoded")
        for c in sintomas
    ],
)

print("\n2️IMPUTACIÓN: Llenar valores faltantes con la mediana de cada columna")
print(f"   (incluye '{NO_RESPONDE}')")
# interpolation='lower' garantiza que la mediana sea un valor existente (0, 0.5 o 1)
# y no un promedio intermedio como 0.25
df_transformed = df_transformed.with_columns(
    pl.col(c).fill_null(pl.col(c).quantile(0.5, interpolation='lower'))
    for c in columnas_numericas
)

print("\n3️VALIDACIÓN: Filtrar registros inválidos (antes de normalizar)")
print("   (Age entre 18-55 años, el rango que cubre la codificación)")
df_transformed = df_transformed.filter(pl.col('Age_numeric').is_between(18, 55))

print("\n4️NORMALIZACIÓN: Z-score (media=0, std=1)")
print("   (Crítico para K-means: Age 18-55 vs síntomas 0-1)")
df_transformed = df_transformed.with_columns(
    ((pl.col(f"{c}_encoded") - pl.col(f"{c}_encoded").mean()) / pl.col(f"{c}_encoded").std())
    .alias(f"{c}_normalized")
    for c in sintomas
).with_columns(
    ((pl.col('Age_numeric') - pl.col('Age_numeric').mean()) / pl.col('Age_numeric').std())
    .alias('Age_normalized')
)

print("\nSELECCIONAR COLUMNAS FINALES")
print("   (Solo features normalizadas + target)")
final_cols = (['Age_normalized'] + [f"{c}_normalized" for c in sintomas]
              + [f"{target_col}_encoded"])
df_transformed = df_transformed.select(final_cols)

print(f"   ✓ Features seleccionadas: {len(final_cols)}")
print("   ✓ Ready para ejecutar")
print("Las transformaciones están DEFINIDAS pero NO ejecutadas")
print("  El optimizador de Polars las ejecutará en paralelo al .collect()")

DEFINIR TRANSFORMACIONES LAZY (aún NO se ejecutan)

1️CONVERSIÓN DE TIPOS: STRING → NÚMERO
   (Age: '35-40' → 37.5 | No/Not at all→0, Sometimes/Maybe/Often/Two or more days→0.5, Yes→1)

2️IMPUTACIÓN: Llenar valores faltantes con la mediana de cada columna
   (incluye 'Not interested to say')

3️VALIDACIÓN: Filtrar registros inválidos (antes de normalizar)
   (Age entre 18-55 años, el rango que cubre la codificación)

4️NORMALIZACIÓN: Z-score (media=0, std=1)
   (Crítico para K-means: Age 18-55 vs síntomas 0-1)

SELECCIONAR COLUMNAS FINALES
   (Solo features normalizadas + target)
   ✓ Features seleccionadas: 11
   ✓ Ready para ejecutar
Las transformaciones están DEFINIDAS pero NO ejecutadas
  El optimizador de Polars las ejecutará en paralelo al .collect()


##Ejecutar transformaciones (.collect())



In [7]:
import time

print("\EJECUTAR PLAN OPTIMIZADO")
print("\Procesando...")
start_time = time.time()
start_memory = psutil.Process().memory_info().rss / (1024**2)

# .collect() ejecuta el plan optimizado
try:
    df_processed = df_transformed.collect()

    elapsed_time = time.time() - start_time
    end_memory = psutil.Process().memory_info().rss / (1024**2)
    peak_memory = end_memory - start_memory

    print(f"\Ejecución completada")
    print(f"Tiempo: {elapsed_time:.2f} segundos")
    print(f"RAM pico: {peak_memory:.1f} MB")
    print(f"Registros procesados: {len(df_processed):,}")
    print(f"Features: {df_processed.shape[1]}")

    print("\Vista previa:")
    print(df_processed.head())

except Exception as e:
    print(f"Error durante ejecución: {e}")
    print("Posible solución:")
    print("Verifica que las columnas existan en el dataset")
    print("Revisa los nombres de columnas en el PASO 4")

<>:3: SyntaxWarning: invalid escape sequence '\E'
<>:4: SyntaxWarning: invalid escape sequence '\P'
<>:16: SyntaxWarning: invalid escape sequence '\E'
<>:22: SyntaxWarning: invalid escape sequence '\V'
<>:3: SyntaxWarning: invalid escape sequence '\E'
<>:4: SyntaxWarning: invalid escape sequence '\P'
<>:16: SyntaxWarning: invalid escape sequence '\E'
<>:22: SyntaxWarning: invalid escape sequence '\V'
/tmp/ipykernel_2791/1609789732.py:3: SyntaxWarning: invalid escape sequence '\E'
  print("\EJECUTAR PLAN OPTIMIZADO")
/tmp/ipykernel_2791/1609789732.py:4: SyntaxWarning: invalid escape sequence '\P'
  print("\Procesando...")
/tmp/ipykernel_2791/1609789732.py:16: SyntaxWarning: invalid escape sequence '\E'
  print(f"\Ejecución completada")
/tmp/ipykernel_2791/1609789732.py:22: SyntaxWarning: invalid escape sequence '\V'
  print("\Vista previa:")


\EJECUTAR PLAN OPTIMIZADO
\Procesando...
\Ejecución completada
Tiempo: 0.25 segundos
RAM pico: 13.8 MB
Registros procesados: 1,503
Features: 11
\Vista previa:
shape: (5, 11)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ Age_norma ┆ Feeling   ┆ Irritable ┆ Trouble   ┆ … ┆ Feeling   ┆ Problems  ┆ Suicide   ┆ Feeling  │
│ lized     ┆ sad or    ┆ towards   ┆ sleeping  ┆   ┆ of guilt_ ┆ of        ┆ attempt_n ┆ sad or   │
│ ---       ┆ Tearful_n ┆ baby &    ┆ at night_ ┆   ┆ normalize ┆ bonding   ┆ ormalized ┆ Tearful_ │
│ f64       ┆ ormali…   ┆ partn…    ┆ norm…     ┆   ┆ d         ┆ with      ┆ ---       ┆ encoded  │
│           ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ baby_…    ┆ f64       ┆ ---      │
│           ┆ f64       ┆ f64       ┆ f64       ┆   ┆ f64       ┆ ---       ┆           ┆ f64      │
│           ┆           ┆           ┆           ┆   ┆           ┆ f64       ┆           ┆          │
╞═══════════╪═════

In [8]:
print("VALIDACIÓN DE CALIDAD")

print(f"\n✓ Registros originales: {len(df_lazy.collect()):,}")
print(f"✓ Registros después limpieza: {len(df_processed):,}")
print(f"✓ Registros eliminados (inválidos): {len(df_lazy.collect()) - len(df_processed):,}")

print(f"\n✓ Features: {df_processed.shape[1]}")
print(f"✓ Valores faltantes: {df_processed.null_count().sum_horizontal()}")
print(f"✓ Duplicados: {len(df_processed) - df_processed.unique().shape[0]}")

print(f"\n✓ Estadísticas de features normalizados:")
print(df_processed.describe())

VALIDACIÓN DE CALIDAD

✓ Registros originales: 1,503
✓ Registros después limpieza: 1,503
✓ Registros eliminados (inválidos): 0

✓ Features: 11
✓ Valores faltantes: shape: (1,)
Series: 'sum' [u32]
[
	0
]
✓ Duplicados: 1178

✓ Estadísticas de features normalizados:
shape: (9, 12)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ statistic ┆ Age_norma ┆ Feeling   ┆ Irritable ┆ … ┆ Feeling   ┆ Problems  ┆ Suicide   ┆ Feeling  │
│ ---       ┆ lized     ┆ sad or    ┆ towards   ┆   ┆ of guilt_ ┆ of        ┆ attempt_n ┆ sad or   │
│ str       ┆ ---       ┆ Tearful_n ┆ baby &    ┆   ┆ normalize ┆ bonding   ┆ ormalized ┆ Tearful_ │
│           ┆ f64       ┆ ormali…   ┆ partn…    ┆   ┆ d         ┆ with      ┆ ---       ┆ encoded  │
│           ┆           ┆ ---       ┆ ---       ┆   ┆ ---       ┆ baby_…    ┆ f64       ┆ ---      │
│           ┆           ┆ f64       ┆ f64       ┆   ┆ f64       ┆ ---       ┆           ┆ f64      │
│           ┆ 

In [9]:
print("GUARDAR EN PARQUET")

output_file = 'data/postpartum_depression_processed.parquet'

print(f"\nGuardando a Parquet...")
print(f"Compresión: snappy (balance velocidad/tamaño)")

start_time = time.time()
df_processed.write_parquet(
    output_file,
    compression='snappy'
)
elapsed_time = time.time() - start_time

parquet_size = os.path.getsize(output_file) / (1024**2)
csv_size = os.path.getsize(csv_path) / (1024**2)
compression_ratio = csv_size / parquet_size

print(f"\nArchivo guardado")
print(f"Ubicación: {output_file}")
print(f"Tamaño CSV: {csv_size:.2f} MB")
print(f"Tamaño Parquet: {parquet_size:.2f} MB")
print(f"Compresión: {compression_ratio:.1f}x más pequeño")
print(f"Tiempo escritura: {elapsed_time:.2f}s")

GUARDAR EN PARQUET

Guardando a Parquet...
Compresión: snappy (balance velocidad/tamaño)

Archivo guardado
Ubicación: data/postpartum_depression_processed.parquet
Tamaño CSV: 0.11 MB
Tamaño Parquet: 0.03 MB
Compresión: 4.2x más pequeño
Tiempo escritura: 0.03s


##Descargar resultado

In [10]:
from google.colab import files

print("DESCARGAR ARCHIVO PROCESADO")

print(f"Preparando descarga...")
files.download('data/postpartum_depression_processed.parquet')

print(f"\nDescarga iniciada")
print(f"Archivo: postpartum_depression_processed.parquet")
print(f"Tamaño: {parquet_size:.2f} MB")
print(f"Formato: Parquet (100x más rápido que CSV)")

DESCARGAR ARCHIVO PROCESADO
Preparando descarga...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Descarga iniciada
Archivo: postpartum_depression_processed.parquet
Tamaño: 0.03 MB
Formato: Parquet (100x más rápido que CSV)


DATA AUMENTATION